<a href="https://colab.research.google.com/github/Jared2562897/practica-rnap/blob/main/CEDE%C3%91O%20VILLOTA%20RICARDO%20JARED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np

def funcion_escalon(x):
    return 1 if x >= 0 else 0

In [15]:
class Perceptron:
    def __init__(self, n_entradas, lr=0.1):
        self.w = np.zeros(n_entradas) # Pesos iniciales en 0
        self.b = 0.0                  # Bias inicial en 0
        self.lr = lr

    def predecir(self, x):
        z = np.dot(x, self.w) + self.b
        return funcion_escalon(z)

    def entrenamiento(self, X, y, epocas=10):
        print(f"{'Época':<7} | {'Entrada':<10} | {'Pesos [w1, w2]':<18} | {'Bias':<8} | {'Error'}")
        print("-" * 65)

        for ep in range(epocas):
            errores_en_epoca = 0
            for i in range(len(X)):
                y_pred = self.predecir(X[i])
                error = y[i] - y_pred

                if error != 0:
                    # Regla de actualización de Rosenblatt:
                    # w = w + lr * (y_real - y_pred) * x
                    self.w += self.lr * error * X[i]
                    self.b += self.lr * error
                    errores_en_epoca += 1

                # Imprimir evolución por cada muestra para ver el cambio paso a paso
                print(f"{ep:<7} | {str(X[i]):<10} | {str(np.round(self.w, 2)):<18} | {self.b:<8.2f} | {error}")

            if errores_en_epoca == 0:
                print(f"\n--- Convergencia lograda en la época {ep} ---")
                break

In [16]:
# Entradas (x1, x2)
X_and = np.array([[0,0], [0,1], [1,0], [1,1]])
# Salidas deseadas para AND
y_and = np.array([0, 0, 0, 1])

# Instanciar y entrenar
modelo_and = Perceptron(n_entradas=2, lr=0.1)
modelo_and.entrenamiento(X_and, y_and)

Época   | Entrada    | Pesos [w1, w2]     | Bias     | Error
-----------------------------------------------------------------
0       | [0 0]      | [0. 0.]            | -0.10    | -1
0       | [0 1]      | [0. 0.]            | -0.10    | 0
0       | [1 0]      | [0. 0.]            | -0.10    | 0
0       | [1 1]      | [0.1 0.1]          | 0.00     | 1
1       | [0 0]      | [0.1 0.1]          | -0.10    | -1
1       | [0 1]      | [0.1 0. ]          | -0.20    | -1
1       | [1 0]      | [0.1 0. ]          | -0.20    | 0
1       | [1 1]      | [0.2 0.1]          | -0.10    | 1
2       | [0 0]      | [0.2 0.1]          | -0.10    | 0
2       | [0 1]      | [0.2 0. ]          | -0.20    | -1
2       | [1 0]      | [0.1 0. ]          | -0.30    | -1
2       | [1 1]      | [0.2 0.1]          | -0.20    | 1
3       | [0 0]      | [0.2 0.1]          | -0.20    | 0
3       | [0 1]      | [0.2 0.1]          | -0.20    | 0
3       | [1 0]      | [0.2 0.1]          | -0.20    | 0
3       | [1 

In [17]:
w1, w2 = modelo_and.w
b = modelo_and.b

print("--- RESULTADOS FINALES ---")
print(f"Ecuación: {w1:.2f}*x1 + {w2:.2f}*x2 + {b:.2f} = 0")

# Despejando x2 para verla como una recta (y = mx + c)
if w2 != 0:
    m = -w1 / w2
    c = -b / w2
    print(f"Frontera de decisión: x2 = {m:.2f}*x1 + ({c:.2f})")
else:
    print("La frontera es una línea vertical.")

--- RESULTADOS FINALES ---
Ecuación: 0.20*x1 + 0.10*x2 + -0.20 = 0
Frontera de decisión: x2 = -2.00*x1 + (2.00)


**EJERCICIO NUMERO 2**




In [18]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    # Restamos el max por estabilidad numérica
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [19]:
class RedNeuronal:
    def __init__(self, n_entradas, n_ocultas, n_salida, lr=0.01):
        # Inicialización de pesos (He initialization para ReLU)
        self.w1 = np.random.randn(n_entradas, n_ocultas) * np.sqrt(2/n_entradas)
        self.b1 = np.zeros((1, n_ocultas))

        self.w2 = np.random.randn(n_ocultas, n_salida) * np.sqrt(2/n_ocultas)
        self.b2 = np.zeros((1, n_salida))

        self.lr = lr

    def forward(self, X):
        # Capa oculta con ReLU
        self.z1 = np.dot(X, self.w1) + self.b1
        self.a1 = relu(self.z1)

        # Capa de salida con Softmax
        self.z2 = np.dot(self.a1, self.w2) + self.b2
        self.a2 = softmax(self.z2)
        return self.a2

    def backward(self, X, y_real):
        m = y_real.shape[0]

        # Gradiente en la salida (Cross-Entropy + Softmax)
        dz2 = self.a2 - y_real
        dw2 = (1/m) * np.dot(self.a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)

        # Gradiente en la capa oculta
        dz1 = np.dot(dz2, self.w2.T) * relu_derivada(self.z1)
        dw1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)

        # Actualización de pesos (Descenso del gradiente)
        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1

    def predecir(self, X):
        # Retorna la clase con mayor probabilidad
        return np.argmax(self.forward(X), axis=1)

    def calcular_perdida(self, y_pred, y_real):
        # Cross-entropy loss
        m = y_real.shape[0]
        # Clipping para evitar log(0)
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return -np.sum(y_real * np.log(y_pred)) / m

In [20]:
# 1. Cargar y preprocesar datos
iris = load_iris()
X, y = iris.data, iris.target

# Escalamiento (Normalización)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# One-hot encoding para las etiquetas (3 clases)
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

# 2. Inicializar Red (4 entradas, 8 neuronas ocultas, 3 salidas)
modelo = RedNeuronal(n_entradas=4, n_ocultas=8, n_salida=3, lr=0.1)

In [21]:
# 3. Entrenamiento
for epoca in range(1001):
    y_pred = modelo.forward(X_train)
    modelo.backward(X_train, y_train)

    if epoca % 200 == 0:
        loss = modelo.calcular_perdida(y_pred, y_train)
        print(f"Época {epoca} - Pérdida: {loss:.4f}")

# 4. Evaluación final
y_pred_test = modelo.predecir(X_test)
y_real_test = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred_test == y_real_test) * 100
print(f"\nAccuracy Final en Test: {accuracy:.2f}%")

# Ejemplo de predicción individual
print("\nPredicciones (Muestra):", y_pred_test[:5])
print("Reales       (Muestra):", y_real_test[:5])

Época 0 - Pérdida: 2.1760
Época 200 - Pérdida: 0.1667
Época 400 - Pérdida: 0.0807
Época 600 - Pérdida: 0.0626
Época 800 - Pérdida: 0.0558
Época 1000 - Pérdida: 0.0524

Accuracy Final en Test: 100.00%

Predicciones (Muestra): [1 0 2 1 1]
Reales       (Muestra): [1 0 2 1 1]
